# Hyperparameter tuning and calibration

For every dataset:

1. Stratified, grouped **80/20 split** (held-out test set, ids saved to `holdout_ids.csv`).
2. **Random search** on the 80 % part with 5-fold `StratifiedGroupKFold`. Candidate 0 is always the current default from `advancedModels.ipynb`. Objective: Integrated Brier Score.
3. Default and tuned setting are refitted on the 80 % and evaluated **once** on the test set.
4. **Calibration** on the test set: decile calibration curves (30/180/365 days), Cox-Snell residuals, PIT Q-Q plot with D-calibration p-value.

All logic lives in `tuning_calibration.py`. Results are written to `tuning_output/<dataset>/`.

In [ ]:
import pandas as pd

import tuning_calibration as tc

pd.set_option("display.max_columns", 50)

In [ ]:
DATASETS = [
    "../combination/daysAfterStart_modelData.csv",
    "../combination/date_modelData.csv",
    "../combination/assignmentsAfterStart_modelData.csv",
]

MODELS = ["GBS", "RSF", "Cox"]

N_CANDIDATES = 20

# Additional columns to exclude from the features. Columns describing the
# state at extraction time (status flags, current pay grade, discontinued
# "ZZ_" values) are always excluded, see tc.snapshot_leakage_columns.
EXTRA_EXCLUDE = None

In [ ]:
test_metrics = []

for dataset in DATASETS:
    test_metrics.append(
        tc.tune_and_evaluate(
            dataset,
            models=MODELS,
            n_candidates=N_CANDIDATES,
            extra_exclude=EXTRA_EXCLUDE,
            output_dir="tuning_output",
        )
    )

test_metrics = pd.concat(test_metrics, ignore_index=True)
test_metrics.to_csv("tuning_output/test_metrics_all.csv", sep=";", index=False)

## Default vs. tuned on the held-out test set

In [ ]:
test_metrics[
    [
        "dataset",
        "model",
        "setting",
        "c_index",
        "integrated_brier_score",
        "brier_365",
        "auc_30",
        "auc_180",
        "auc_365",
        "calibration_365",
        "d_calibration_p",
    ]
].round(4)

## Best settings

In [ ]:
test_metrics.loc[
    test_metrics["setting"] == "tuned",
    ["dataset", "model", "params"]
]